In [55]:
!pip install river
!pip install deepctr-torch

In [56]:
import numpy as np
np.float = float
from river.drift import ADWIN
import random
import os
import pandas as pd
import matplotlib.pyplot as plt

import torch
import keras
import json

import sklearn
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix

from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.sequence import pad_sequences

from deepctr_torch.inputs import SparseFeat, VarLenSparseFeat, get_feature_names
from deepctr_torch.models import DeepFM

In [ ]:
#loading the data
reviews = pd.read_json("/Users/jasperbruin/Documents/driftwatch/LAB experiments/datasets/CDs_and_Vinyl.jsonl", lines=True)

In [ ]:
#preparing the data
reviews = reviews[reviews["rating"] != 3]
reviews = reviews[reviews["timestamp"] > "2015-01-01"]
reviews["year"] = reviews["timestamp"].apply(lambda x: x.year)
reviews["month"] = reviews["timestamp"].apply(lambda x: x.month)
reviews["day"] = reviews["timestamp"].apply(lambda x: x.day)

In [ ]:
#create reference dataframe
reference_data = reviews[reviews["timestamp"] > "2015-01-01"]
reference_data = reference_data[reference_data["timestamp"] < "2019-01-01"]
#create evaluation dataframe
evaluation_data = reviews[reviews["timestamp"] >= "2019-01-01"]
evaluation_data = evaluation_data.set_index(evaluation_data["timestamp"])
evaluation_data.sort_index(inplace = True)

In [ ]:
# remove products that appear after ref
evaluation_data_with_old_products = evaluation_data[evaluation_data["parent_asin"].isin(reference_data["parent_asin"])]

In [ ]:
#encode features in the reviews for use in ml model
sparse_features = ["parent_asin", "user_id","year","month","day"]
for feat in sparse_features:
        lbe = LabelEncoder()
        reviews[feat] = lbe.fit_transform(reviews[feat])

In [ ]:
def create_matrix(df):
	
	N = len(df['user_id'].unique())
	M = len(df['product_id'].unique())
	
	# Map Ids to indices
	user_mapper = dict(zip(np.unique(df["user_id"]), list(range(N))))
	product_mapper = dict(zip(np.unique(df["product_id"]), list(range(M))))
	
	# Map indices to IDs
	user_inv_mapper = dict(zip(list(range(N)), np.unique(df["user_id"])))
	product_inv_mapper = dict(zip(list(range(M)), np.unique(df["product_id"])))
	
	user_index = [user_mapper[i] for i in df['user_id']]
	product_index = [product_mapper[i] for i in df['product_id']]

	X = csr_matrix((df["rating"], (product_index, user_index)), shape=(M, N))
	
	return X, user_mapper, product_mapper, user_inv_mapper, product_inv_mapper

In [ ]:
#rename for use in function
reference_data["product_id"] = reference_data["parent_asin"]

In [ ]:
X, user_mapper, product_mapper, user_inv_mapper, product_inv_mapper = create_matrix(reference_data)

In [ ]:
#train ml model
kNN = NearestNeighbors(n_neighbors=5, algorithm="brute", metric='cosine')
kNN.fit(X)

In [ ]:
def find_similar_products(product_id, X, k, metric='cosine', show_distance=False):

  neighbour_ids = []

  product_ind = product_mapper[product_id]
  product_vec = X[product_ind]
  k+=1
  product_vec = product_vec.reshape(1,-1)
  neighbour = kNN.kneighbors(product_vec, return_distance=show_distance)
  for i in range(0,k):
    n = neighbour.item(i)
    neighbour_ids.append(product_inv_mapper[n])
  neighbour_ids.pop(0)
  return neighbour_ids

In [ ]:
#sample reviews from products
# sampled = evaluation_data_with_old_products.sample(100)

sampled = evaluation_data_with_old_products.sample(10000)

In [ ]:
#save sample
sampled.reset_index(inplace=True,drop=True)
sampled.to_json("sampled_products.json")

In [ ]:
#store the predictions of the recommendation system (takes a long time)
preds = []
for i in range(len(sampled)):
    preds.append(find_similar_products(sampled["parent_asin"].values[i],X,4))

In [ ]:
sampled.sort_index(inplace = True)

In [ ]:
#store the recommendations in dataframe
sampled["recommendation"] = preds

In [ ]:
reviews_copy = pd.read_json("/Users/jasperbruin/Documents/driftwatch/LAB experiments/datasets/CDs_and_Vinyl.jsonl", lines=True)

In [ ]:
#ensure same strutcture as ctr-predictor training data
reviews_copy = reviews_copy[reviews_copy["rating"] != 3]
reviews_copy = reviews_copy[reviews_copy["timestamp"] > "2015-01-01"]

In [ ]:
# create a mapper for the ctr-predictor
music_mapper = dict(zip(reviews_copy["parent_asin"].unique(), reviews["parent_asin"].unique()))

In [ ]:
# create a mapper for the ctr-predictor
music_user_mapper = dict(zip(reviews_copy["user_id"].unique(), reviews["user_id"].unique()))

In [ ]:
# load the ctr-predictor
model = torch.load("/Users/jasperbruin/Documents/driftwatch/LAB experiments/models/model_formatted_music_v2_cpu")

In [ ]:
# calculate click-through-rate for the recommendations
def mean_predicted_ctr():
    res = []
    for i in range(len(sampled)):
        temp = []
        for j in range(len(sampled["recommendation"][i])):
            temp.append(model.predict({"parent_asin": pd.Series(music_mapper[sampled["recommendation"][i][j]]),"user_id": pd.Series(music_user_mapper[sampled["user_id"][i]]), "year": pd.Series(sampled["year"][i] - 2015), "month": pd.Series(sampled["month"][i] - 1), "day": pd.Series(sampled["day"][i] - 1)}))
        res.append(temp)    
    return res

In [ ]:
ctr_list = mean_predicted_ctr()

In [ ]:
ctr_list

In [ ]:
means = []
for i in range(len(ctr_list)):
    means.append(np.mean(ctr_list[i]))

In [ ]:
np.mean(means)

In [ ]:
series_means = pd.Series(means, index=sampled["timestamp"])
series_means = series_means.sort_index()

In [ ]:
rolling_means = series_means.rolling("30D").mean()

In [ ]:
rolling_means.plot()

In [ ]:
# create labels for the drift-detectors
correct_recommendations = []
for i in range(len(ctr_list)):
    for j in range(4):
        correct_recommendations.append(random.choices([0,1],weights=[1-ctr_list[i][j][0][0],ctr_list[i][j][0][0]])[0])

In [ ]:
#testing the different erb-detectors

In [ ]:
adwin = ADWIN()
entry_number = 0
entries = []

for i in range(len(correct_recommendations)):
    # Update the detector with the current value
    adwin.update(correct_recommendations[i])
    
    # Check if drift is detected
    if adwin.drift_detected:
        print(
            f"Change detected at index {i}, value: {correct_recommendations[i]}, entry: {entry_number}"
        )
        entries.append(entry_number)
    
    # Update entry number every 4 iterations
    if i % 4 == 3:
        entry_number += 1


In [ ]:
for i in entries:
    print(sampled["timestamp"][i])

In [ ]:
np.datetime64("2019-06-01") +  296 * np.timedelta64(1, 'D')

In [ ]:
np.datetime64('2020-07-07') - np.datetime64('2020-03-23')

In [ ]:
counter = 0
for i in entries:
    print(sampled["timestamp"][i])
    if np.datetime64(sampled["timestamp"][i]) >= np.datetime64('2020-03-23'):
        counter = counter + 1

In [ ]:
counter